In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import warnings
warnings.filterwarnings("ignore")
# 提供一个物种Taxonomy ID的列表
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-12-03 TE HT Draft\\Tables\\'
samples=pd.read_excel(table_path+"Table S1\\Table S1.Sample_Information.xlsx").fillna("")
#samples['CODE_ID']=samples['Specie_ID']
taxid_dic={}
for i in range(samples.shape[0]):
    taxid_dic[samples.loc[i,'Specie_ID']]=samples.loc[i,'Taxonomy_ID']
samples.head(1)

,Name,Specie_ID,Taxonomy_ID,Assembly_Accession,Assembly,Genome_Source,Plant_Group,subphylum,class,order,family,genus
0,Acer negundo,PBXX,4023,GCA_025594385.1,ASM2559438v1,NCBI,Dicots,Streptophytina,Magnoliopsida,Sapindales,Sapindaceae,Acer


### step1.background genes
- 整理每个样本 HT相关家族上下游10kb范围内的基因

In [3]:
data=pd.read_excel('D:\\19.TE_HT\\Work\\HTTs_Species_Pairs_Pos_Stats_With_SeqCounts.V2.xlsx')
data=data[(data['Source_Seq_Count']>0)&(data['Target_Seq_Count']>0)]
print(data.shape,len(list(data['Group'].unique())))
data.head(1)

(49710, 11) 686


,Family,Group,HT_Target,HT_Source,HT_Sim,Target_Species,Source_Species,Source_Species_id,Target_Species_id,Source_Seq_Count,Target_Seq_Count
0,Gypsy,TE_Gypsy_OG0021796,BUJI,FVXP,0.999645,Taxus chinensis,Oryza sativa DG,4530,29808,1639,6050


In [5]:
data1=data[data['Group'].apply(lambda x:1 if x.startswith("TE_") else 0)==1]
data2=data[data['Group'].apply(lambda x:1 if x.startswith("TEpep_") else 0)==1]

In [ ]:
te_bed_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'
tepep_bed_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'
bg_gene_path='D:\\19.TE_HT\\07.Enrichment\\1.Background_Genes\\'
htt_samples=list(set(data1['HT_Target'].tolist()+data1['HT_Source'].tolist()))
httpep_samples=list(set(data2['HT_Target'].tolist()+data2['HT_Source'].tolist()))
len(htt_samples)
for sample in samples['Specie_ID'].tolist():
    genes=[]
    if sample in htt_samples:
        bed=pd.read_csv(te_bed_path+sample+"_TE.bed",sep='\t').fillna("")
        bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
        genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
    if sample in httpep_samples:
        bed=pd.read_csv(tepep_bed_path+sample+"_TEpep.bed",sep='\t').fillna("")
        bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
        genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
    genes=list(set(genes))
    if sample in rice_samples:
        genes=get_rice_gene_names(genes)
    if sample in maize_samples:
        genes=get_maize_gene_names(genes)
    print(sample,len(genes))
    if len(genes)>0:
        with open(bg_gene_path+sample+"_BG_Genes.txt",'w')as f:
            for gene in genes:
                f.write(gene+"\n")
            f.close()

### 生成gene lists

### 按TE家族生成list

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\3.Family_Genes\\'
for f,f_df in data.groupby("Group"):
    f_samples=list(set(f_df['HT_Target'].tolist()+f_df['HT_Source'].tolist()))
    f_type=f.split("_")[0]
    if f_type=='TE':
        f_genes=[]
        for sample in f_samples:
            bed=pd.read_csv(te_bed_path+sample+"_TE.bed",sep='\t').fillna("")
            bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
            bed=bed[(bed['Orthogroup']==f)&(bed['Overlap_Gene']!='')]
            if bed.shape[0]>0:
                genes=list(set(" | ".join(bed['Overlap_Gene'].tolist()).split(" | ")))
                if sample in rice_samples:
                    genes=get_rice_gene_names(genes)
                if sample in maize_samples:
                    genes=get_maize_gene_names(genes)
                f_genes+=genes
    if f_type=='TEpep':
        f_genes=[]
        for sample in f_samples:
            bed=pd.read_csv(tepep_bed_path+sample+"_TEpep.bed",sep='\t').fillna("")
            bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
            bed=bed[(bed['Orthogroup']==f)&(bed['Overlap_Gene']!='')]
            if bed.shape[0]>0:
                genes=list(set(" | ".join(bed['Overlap_Gene'].tolist()).split(" | ")))
                if sample in rice_samples:
                    genes=get_rice_gene_names(genes)
                if sample in maize_samples:
                    genes=get_maize_gene_names(genes)
                f_genes+=genes
    if len(f_genes)>0:
        with open(save_path+f+'_HT_Genes.txt','w') as f0:
            for gene in f_genes:
                f0.write(gene+"\n")
            f0.close()
    print(f,len(f_genes))

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\5.Plant_Family\\'
for family,f_df in samples.groupby("family"):
    f_samples=f_df['Specie_ID'].tolist()
    #print(family,f_samples)
    if family!='':
        f_genes=[]
        for sample in f_samples:
            genes=[]
            if sample in htt_samples:
                bed=pd.read_csv(te_bed_path+sample+"_TE.bed",sep='\t').fillna("")
                bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
                genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
            if sample in httpep_samples:
                bed=pd.read_csv(tepep_bed_path+sample+"_TEpep.bed",sep='\t').fillna("")
                bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
                genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
            genes=list(set(genes))
            if sample in rice_samples:
                genes=get_rice_gene_names(genes)
            if sample in maize_samples:
                genes=get_maize_gene_names(genes)
            f_genes+=genes
        if len(f_genes)>0:
            with open(save_path+family+'_HT_Genes.txt','w') as f0:
                for gene in f_genes:
                    f0.write(gene+"\n")
                f0.close()
        print(family,len(f_genes))

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Species\\'
for taxon_id,t_df in samples.groupby("Taxonomy_ID"):
    f_samples=t_df['Specie_ID'].tolist()
    species=" ".join(t_df['Name'].tolist()[0].split(" ")[:2])
    #print(family,f_samples)
    if species!='':
        f_genes=[]
        for sample in f_samples:
            genes=[]
            if sample in htt_samples:
                bed=pd.read_csv(te_bed_path+sample+"_TE.bed",sep='\t').fillna("")
                bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
                genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
            if sample in httpep_samples:
                bed=pd.read_csv(tepep_bed_path+sample+"_TEpep.bed",sep='\t').fillna("")
                bed['Overlap_Gene']=bed['Overlap_Gene'].apply(lambda x:str(x))
                genes+=" | ".join(bed[bed['Overlap_Gene']!='']['Overlap_Gene'].tolist()).split(" | ")
            genes=list(set(genes))
            if sample in rice_samples:
                genes=get_rice_gene_names(genes)
            if sample in maize_samples:
                genes=get_maize_gene_names(genes)
            f_genes+=genes
        if len(f_genes)>0:
            with open(save_path+species+'_HT_Genes.txt','w') as f0:
                for gene in f_genes:
                    f0.write(gene+"\n")
                f0.close()
        print(species,len(f_genes))

### Enrichment

In [3]:
import gseapy as gp
from gseapy.plot import barplot, dotplot

In [4]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\'
go_gene_sets={}
go_path='D:\\18.TE_Evolution\\03.Protein_Annotation\\GO\\'
for i in range(samples.shape[0]):
    if i%50==0:
        print(i)
    sample=samples.loc[i,'Specie_ID']
    if sample+"_BG_Genes.txt" in os.listdir("D:\\19.TE_HT\\07.Enrichment\\1.Background_Genes\\"):
        bg_genes=[gene.strip() for gene in open("D:\\19.TE_HT\\07.Enrichment\\1.Background_Genes\\"+ sample+"_BG_Genes.txt",'r').readlines()]
        if sample+".wego.csv" in os.listdir(go_path):
            df=pd.read_csv(go_path+sample+".wego.csv")
            df=df.dropna().reset_index(drop=True)
            for i in range(df.shape[0]):
                gene=df.loc[i,'Gene']
                if gene in bg_genes:
                    gos=df.loc[i,'Go_Terms']
                    if "\t" in gos:
                        gos=gos.split("\t")
                    elif " " in gos:
                        gos=gos.split(" ")
                    else:
                        gos=[gos]
                    for go in gos:
                        if go in go_gene_sets:
                            go_gene_sets[go].append(gene)
                        else:
                            go_gene_sets[go]=[gene]

0
50
100
150
200
250
300
350
400
450
500
550


In [5]:
go_titles=pd.read_excel("D:\\18.TE_Evolution\\03.Protein_Annotation\\GO_Titles.xlsx")
go_title_dic={}
for i in range(go_titles.shape[0]):
    go_title_dic[go_titles.loc[i,'index']]=go_titles.loc[i,'Title']

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\TE_Family\\'
gene_list_path='D:\\19.TE_HT\\07.Enrichment\\3.Family_Genes\\'
files=os.listdir(gene_list_path)

for file in files[562:]:
    gene_list=[gene.strip() for gene in open(gene_list_path+file,'r').readlines()]
    family=file.split("_HT_")[0]
    print(file,len(gene_list))
    if len(gene_list)>4:
        if file.replace(".txt","_GO.xlsx") not in os.listdir(save_path):
            try:
                enr = gp.enrichr(gene_list=gene_list,#所需查询gene_list，可以是一个列表，也可为文件（一列，每行一个基因）
                                gene_sets=go_gene_sets,#gene set library，多个相关的gene set 。如所有GO term组成一个gene set library.
                                outdir=save_path,#输出目录
                                top_term=20,
                                cutoff=0.01#pvalue阈值
                                                 )
                result=enr.results
                result=result.sort_values("Adjusted P-value")
                #for col in ['GO_Type',"Title","Level"]:
                result["Title"]=result['Term'].apply(lambda x:go_title_dic[x] if x in go_title_dic else '')
                print(family,result.shape)
                result.to_excel(save_path+file.replace(".txt","_GO.xlsx"),index=False)
            except Exception as e:
                print(e)
                pass
    else:
        print(file,family)

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\Plant_Family\\'
gene_list_path='D:\\19.TE_HT\\07.Enrichment\\5.Plant_Family\\'
for file in os.listdir(gene_list_path):
    gene_list=[gene.strip() for gene in open(gene_list_path+file,'r').readlines()]
    family=file.split("_HT_")[0]
    print(file,len(gene_list))
    if len(gene_list)>4:
        try:
            enr = gp.enrichr(gene_list=gene_list,#所需查询gene_list，可以是一个列表，也可为文件（一列，每行一个基因）
                            gene_sets=go_gene_sets,#gene set library，多个相关的gene set 。如所有GO term组成一个gene set library.
                            outdir=save_path,#输出目录
                            top_term=20,
                            cutoff=0.01#pvalue阈值
                                             )
            result=enr.results
            result=result.sort_values("Adjusted P-value")
            #for col in ['GO_Type',"Title","Level"]:
            result["Title"]=result['Term'].apply(lambda x:go_title_dic[x] if x in go_title_dic else '')
            print(family,result.shape)
            result.to_excel(save_path+file.replace(".txt","_GO.xlsx"),index=False)
        except:
            pass
    else:
        print(file,family)

In [52]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\Plant_Groups\\'
gene_list_path='D:\\19.TE_HT\\07.Enrichment\\6.Plant_Group\\'
for file in os.listdir(gene_list_path):
    gene_list=[gene.strip() for gene in open(gene_list_path+file,'r').readlines()]
    family=file.split("_HT_Genes")[0]
    print(file,len(gene_list))
    if len(gene_list)>4:
        try:
            enr = gp.enrichr(gene_list=gene_list,#所需查询gene_list，可以是一个列表，也可为文件（一列，每行一个基因）
                            gene_sets=go_gene_sets,#gene set library，多个相关的gene set 。如所有GO term组成一个gene set library.
                            outdir=save_path,#输出目录
                            top_term=20,
                            cutoff=0.01#pvalue阈值
                                             )
            result=enr.results
            result=result.sort_values("Adjusted P-value")
            #for col in ['GO_Type',"Title","Level"]:
            result["Title"]=result['Term'].apply(lambda x:go_title_dic[x] if x in go_title_dic else '')
            print(family,result.shape)
            result.to_excel(save_path+file.replace(".txt","_GO.xlsx"),index=False)
        except:
            pass
    else:
        print(file,family)

Algae_HT_Genes.txt 365
Algae (189, 9)
Dicots_HT_Genes.txt 1635200
Dicots (7860, 9)
Ferns_HT_Genes.txt 32884
Ferns (1910, 9)
Gymnosperms_HT_Genes.txt 15389
Gymnosperms (1330, 9)
Monocots_HT_Genes.txt 693378
Monocots (2688, 9)
Mosses_HT_Genes.txt 344
Mosses (199, 9)


In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\Plant_Species\\'
gene_list_path='D:\\19.TE_HT\\07.Enrichment\\2.Plant_Species\\'
for file in os.listdir(gene_list_path):
    gene_list=[gene.strip() for gene in open(gene_list_path+file,'r').readlines()]
    specie_name=file.split('_HT')[0]
    print(specie_name,len(gene_list))
    if len(gene_list)>4:
        go_gene_sets={}
        species_samples=samples[samples['Name'].apply(lambda x:1 if specie_name in x else 0)==1]['Specie_ID'].tolist()
        for sample in species_samples:
            if sample+".wego.csv" in os.listdir(go_path):
                df=pd.read_csv(go_path+sample+".wego.csv")
                df=df.dropna().reset_index(drop=True)
                for i in range(df.shape[0]):
                    gene=df.loc[i,'Gene']
                    gos=df.loc[i,'Go_Terms']
                    if "\t" in gos:
                        gos=gos.split("\t")
                    elif " " in gos:
                        gos=gos.split(" ")
                    else:
                        gos=[gos]
                    for go in gos:
                        if go in go_gene_sets:
                            go_gene_sets[go].append(gene)
                        else:
                            go_gene_sets[go]=[gene]
        try:
            enr = gp.enrichr(gene_list=gene_list,#所需查询gene_list，可以是一个列表，也可为文件（一列，每行一个基因）
                            gene_sets=go_gene_sets,#gene set library，多个相关的gene set 。如所有GO term组成一个gene set library.
                            outdir=save_path,#输出目录
                            top_term=20,
                            cutoff=0.01#pvalue阈值
                                             )
            result=enr.results
            result=result.sort_values("Adjusted P-value")
            #for col in ['GO_Type',"Title","Level"]:
            result["Title"]=result['Term'].apply(lambda x:go_title_dic[x] if x in go_title_dic else '')
            print(specie_name,result.shape)
            result.to_excel(save_path+file.replace(".txt","_GO.xlsx"),index=False)
        except:
            pass
    else:
        print(file,specie_name)

In [ ]:
save_path='D:\\19.TE_HT\\07.Enrichment\\4.Results\\Plant_Family\\'
gene_list_path='D:\\19.TE_HT\\07.Enrichment\\5.Plant_Family\\'
for file in os.listdir(gene_list_path):
    gene_list=[gene.strip() for gene in open(gene_list_path+file,'r').readlines()]
    family=file.split("_")[0]
    print(file,len(gene_list))
    if len(gene_list)>10:
        try:
            enr = gp.enrichr(gene_list=gene_list,#所需查询gene_list，可以是一个列表，也可为文件（一列，每行一个基因）
                            gene_sets=go_gene_sets,#gene set library，多个相关的gene set 。如所有GO term组成一个gene set library.
                            outdir=save_path,#输出目录
                            top_term=20,
                            cutoff=0.01#pvalue阈值
                                             )
            result=enr.results
            result=result.sort_values("Adjusted P-value")
            #for col in ['GO_Type',"Title","Level"]:
            result["Title"]=result['Term'].apply(lambda x:go_title_dic[x] if x in go_title_dic else '')
            print(family,result.shape)
            result.to_excel(save_path+file.replace(".txt","_GO.xlsx"),index=False)
        except:
            pass
    else:
        print(file,family)